# KidLearn Story Model Training (v2 - Improved Quality)

Fine-tune a small language model on the **TinyStories dataset** (Microsoft Research) to generate personalized children's storybooks.

**Base Model**: `roneneldan/TinyStories-33M` (33M parameters, pre-trained on TinyStories)

**Improvements over v1**:
- 100k training stories (up from 50k)
- 5 training epochs (up from 3)
- Added `Loves:` field so the model learns to incorporate the child's interests
- Better data filtering (min 150 chars, min 4 sentences, ASCII-only)
- `<|end|>` as EOS token so the model stops cleanly
- Lower learning rate (3e-5) for better grammar

**Requirements**: Google Colab with GPU (free T4 is sufficient, ~60-90 min training)

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate tokenizers huggingface_hub peft
!pip install -q torch --upgrade

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 2: Load the TinyStories Dataset

In [ ]:
from datasets import load_dataset

print("Loading TinyStories dataset (this may take a few minutes)...")
dataset = load_dataset("roneneldan/TinyStories", split="train")

print(f"Total stories: {len(dataset):,}")
print(f"\nSample story:\n{dataset[0]['text'][:500]}")

## Step 3: Format Stories into Structured Storybooks

We convert raw TinyStories text into a structured format with:
- Title
- Category (adventure, animals, fantasy, moral, bedtime, alphabet)
- Child's name (extracted from story)
- Loves (interest/detail extracted from story keywords)
- 4-6 pages of story content
- Moral/lesson

Quality filters: min 150 chars, max 1500 chars, ASCII-only, min 4 sentences.

In [ ]:
import re
import random

CATEGORIES = ["adventure", "animals", "fantasy", "moral", "bedtime", "alphabet"]

CATEGORY_KEYWORDS = {
    "animals": ["dog", "cat", "bird", "rabbit", "fish", "bear", "lion", "elephant",
                "monkey", "duck", "frog", "turtle", "bunny", "puppy", "kitten",
                "horse", "cow", "pig", "sheep", "chicken", "mouse", "deer", "fox"],
    "fantasy": ["magic", "fairy", "dragon", "wizard", "princess", "prince", "castle",
                "unicorn", "spell", "enchanted", "wand", "kingdom", "wish", "flying"],
    "adventure": ["adventure", "explore", "journey", "discover", "treasure", "map",
                  "forest", "mountain", "brave", "quest", "travel", "climb", "river"],
    "bedtime": ["sleep", "dream", "night", "bed", "moon", "star", "tired", "yawn",
                "pillow", "blanket", "quiet", "soft", "lullaby", "goodnight"],
    "moral": ["learn", "lesson", "share", "kind", "help", "friend", "sorry", "thank",
              "honest", "brave", "try", "important", "truth", "forgive", "fair"],
}

DETAIL_BY_CATEGORY = {
    "animals": ["dogs", "cats", "birds", "rabbits", "lions", "elephants",
                "butterflies", "fish", "horses", "bears", "turtles", "frogs"],
    "fantasy": ["magic", "dragons", "unicorns", "fairies", "castles",
                "wizards", "flying", "rainbows", "stars", "wishes"],
    "adventure": ["exploring", "treasure", "mountains", "the ocean",
                  "rockets", "dinosaurs", "caves", "pirates", "space"],
    "bedtime": ["the moon", "stars", "clouds", "teddy bears",
                "blankets", "dreams", "lullabies"],
    "moral": ["sharing", "kindness", "helping others", "being brave",
              "telling the truth", "friendship", "being fair"],
    "alphabet": ["letters", "reading", "books", "writing", "words", "school"],
}

CHILD_NAMES = ["Lily", "Tom", "Sara", "Ben", "Mia", "Sam", "Emma", "Tim", "Anna", "Max",
               "Abebe", "Hana", "Dawit", "Liya", "Yonas", "Amara", "Kidus", "Selam",
               "Noah", "Olivia", "Ethan", "Sophia", "Aiden", "Zara", "Leo", "Chloe"]

STOPWORDS = {"The", "One", "Once", "There", "They", "She", "But", "And", "His", "Her",
             "This", "That", "After", "Then", "When", "What", "All", "Some", "Every",
             "For", "Not", "Day", "Now", "Little", "Big", "So", "He", "It", "My",
             "How", "Soon", "From", "Just", "As", "In", "On", "At", "To", "Up"}


def is_clean_english(text):
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / max(len(text), 1)
    return ascii_ratio > 0.95


def detect_category(text):
    text_lower = text.lower()
    scores = {}
    for cat, keywords in CATEGORY_KEYWORDS.items():
        scores[cat] = sum(1 for kw in keywords if kw in text_lower)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else random.choice(CATEGORIES)


def extract_detail(text, category):
    text_lower = text.lower()
    keywords = CATEGORY_KEYWORDS.get(category, [])
    found = [kw for kw in keywords if kw in text_lower]
    if found:
        return random.choice(found)
    details = DETAIL_BY_CATEGORY.get(category, ["adventures"])
    return random.choice(details)


def extract_name(text):
    common_names = re.findall(r'\b([A-Z][a-z]{2,8})\b', text[:300])
    names = [n for n in common_names if n not in STOPWORDS]
    return names[0] if names else random.choice(CHILD_NAMES)


def extract_moral(text):
    sentences = re.split(r'[.!?]+', text)
    moral_keywords = ["learned", "lesson", "important", "always", "never", "remember",
                      "realized", "understood", "knew that", "from that day", "happy"]
    for sent in reversed(sentences):
        sent = sent.strip()
        if any(kw in sent.lower() for kw in moral_keywords) and 15 < len(sent) < 150:
            return sent.strip()
    for sent in reversed(sentences):
        sent = sent.strip()
        if 15 < len(sent) < 120:
            return sent.strip()
    return "Every day is a new adventure."


def split_into_pages(text, target_pages=5):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if len(sentences) < 4:
        return []

    per_page = max(2, len(sentences) // target_pages)
    pages = []
    for i in range(0, len(sentences), per_page):
        page = " ".join(sentences[i:i + per_page])
        if len(page.strip()) > 15:
            pages.append(page.strip())

    return pages[:6]


def format_storybook(text):
    text = text.strip()
    if len(text) < 150 or len(text) > 1500:
        return None
    if not is_clean_english(text):
        return None

    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
    if len(sentences) < 4:
        return None

    name = extract_name(text)
    category = detect_category(text)
    detail = extract_detail(text, category)
    pages = split_into_pages(text)
    moral = extract_moral(text)

    if len(pages) < 4:
        return None

    first_sent = text.split('.')[0].strip()
    title = first_sent if len(first_sent) < 60 else f"{name}'s {category.title()} Story"

    formatted = f"<|begin|>\n"
    formatted += f"Title: {title}\n"
    formatted += f"Category: {category}\n"
    formatted += f"Child: {name}\n"
    formatted += f"Loves: {detail}\n\n"
    for i, page in enumerate(pages, 1):
        formatted += f"Page {i}: {page}\n\n"
    formatted += f"Moral: {moral}\n"
    formatted += f"<|end|>"

    return formatted


print("Formatting stories into storybook structure...")
print("Target: 100,000 stories with strict quality filtering\n")

NUM_TO_SAMPLE = 200000
indices = random.sample(range(len(dataset)), min(NUM_TO_SAMPLE, len(dataset)))

formatted_stories = []
TARGET = 100000
for idx in indices:
    if len(formatted_stories) >= TARGET:
        break
    result = format_storybook(dataset[idx]['text'])
    if result:
        formatted_stories.append(result)

print(f"Successfully formatted {len(formatted_stories):,} storybooks")
cat_counts = {}
for s in formatted_stories[:1000]:
    for cat in CATEGORIES:
        if f"Category: {cat}" in s:
            cat_counts[cat] = cat_counts.get(cat, 0) + 1
print(f"Category distribution (first 1k): {cat_counts}")
print(f"\n{'='*60}")
print(f"Sample formatted storybook:")
print(f"{'='*60}")
print(formatted_stories[0])

## Step 4: Load Pre-trained TinyStories Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "roneneldan/TinyStories-33M"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

SPECIAL_TOKENS = ["<|begin|>", "<|end|>"]
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
model.resize_token_embeddings(len(tokenizer))

end_token_id = tokenizer.convert_tokens_to_ids("<|end|>")
print(f"<|end|> token ID: {end_token_id}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Vocab size: {len(tokenizer)}")

## Step 5: Tokenize the Formatted Dataset

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512

story_dataset = Dataset.from_dict({"text": formatted_stories})

def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = story_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
)

split = tokenized_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train samples: {len(train_dataset):,}")
print(f"Eval samples: {len(eval_dataset):,}")

## Step 6: Fine-tune the Model

Training on Colab free T4 GPU takes ~60-90 minutes for 5 epochs on 100k stories. The lower learning rate (3e-5) and more epochs produce better grammar and coherence.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "/content/kidlearn-story-model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=800,
    lr_scheduler_type="cosine",
    logging_steps=200,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=2,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training (5 epochs, lr=3e-5)...")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.0f} seconds")

## Step 7: Test Story Generation

In [ ]:
def generate_story(category, child_name, detail="", max_length=450, temperature=0.8):
    prompt = f"<|begin|>\nTitle:"
    if category:
        prompt = f"<|begin|>\nCategory: {category}\nChild: {child_name}\n\nPage 1:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=temperature,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|end|>"),
        )

    generated = tokenizer.decode(output[0], skip_special_tokens=False)
    return generated


def parse_generated_story(text):
    """Parse the structured output into a dict with title, pages, moral."""
    result = {"title": "", "pages": [], "moral": "", "category": "", "childName": ""}

    text = text.split("<|end|>")[0]
    text = text.replace("<|begin|>", "").strip()

    lines = text.split("\n")
    for line in lines:
        line = line.strip()
        if line.startswith("Title:"):
            result["title"] = line[6:].strip()
        elif line.startswith("Category:"):
            result["category"] = line[9:].strip()
        elif line.startswith("Child:"):
            result["childName"] = line[6:].strip()
        elif line.startswith("Page"):
            page_text = line.split(":", 1)[1].strip() if ":" in line else ""
            if page_text:
                result["pages"].append(page_text)
        elif line.startswith("Moral:"):
            result["moral"] = line[6:].strip()

    if not result["title"]:
        result["title"] = f"{result['childName']}'s Story"

    return result


print("=" * 60)
print("TEST 1: Adventure story for Abebe")
print("=" * 60)
raw = generate_story("adventure", "Abebe")
story1 = parse_generated_story(raw)
print(f"Title: {story1['title']}")
for i, page in enumerate(story1['pages'], 1):
    print(f"Page {i}: {page}")
print(f"Moral: {story1['moral']}")

print(f"\n{'=' * 60}")
print("TEST 2: Animals story for Liya who loves cats")
print("=" * 60)
raw = generate_story("animals", "Liya", "cats")
story2 = parse_generated_story(raw)
print(f"Title: {story2['title']}")
for i, page in enumerate(story2['pages'], 1):
    print(f"Page {i}: {page}")
print(f"Moral: {story2['moral']}")

print(f"\n{'=' * 60}")
print("TEST 3: Bedtime story for Dawit")
print("=" * 60)
raw = generate_story("bedtime", "Dawit")
story3 = parse_generated_story(raw)
print(f"Title: {story3['title']}")
for i, page in enumerate(story3['pages'], 1):
    print(f"Page {i}: {page}")
print(f"Moral: {story3['moral']}")

## Step 8: Save the Final Model

In [ ]:
FINAL_DIR = "/content/kidlearn-story-model-final"

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

import os
total_size = sum(
    os.path.getsize(os.path.join(FINAL_DIR, f))
    for f in os.listdir(FINAL_DIR)
    if os.path.isfile(os.path.join(FINAL_DIR, f))
)
print(f"Model saved to: {FINAL_DIR}")
print(f"Total size: {total_size / 1e6:.1f} MB")
print(f"Files:")
for f in sorted(os.listdir(FINAL_DIR)):
    size = os.path.getsize(os.path.join(FINAL_DIR, f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## Step 9: Save to Google Drive (Backup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/KidLearn-Model"
!mkdir -p "{DRIVE_DIR}"
!cp -r {FINAL_DIR}/* "{DRIVE_DIR}/"

print(f"Model backed up to Google Drive: {DRIVE_DIR}")

## Step 10: Upload to HuggingFace Hub

This lets you use the free HuggingFace Inference API from your app's backend.

1. Create a free account at https://huggingface.co
2. Create an access token at https://huggingface.co/settings/tokens (Write permission)
3. Replace `YOUR_HF_USERNAME` below with your username

In [ ]:
from huggingface_hub import login, HfApi

# Paste your HuggingFace token here
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # <-- REPLACE THIS
HF_USERNAME = "YOUR_USERNAME"     # <-- REPLACE THIS
REPO_NAME = "kidlearn-story-model"

login(token=HF_TOKEN)

REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)

print(f"\nModel uploaded to: https://huggingface.co/{REPO_ID}")
print(f"\nYou can now use this model from your app backend!")
print(f"Set HF_MODEL_ID={REPO_ID} in your backend .env file")

## Step 11: Quick Test with HuggingFace Inference API

Test that the uploaded model works via the API (this is what the app backend will use).

In [ ]:
import requests

API_URL = f"https://api-inference.huggingface.co/models/{REPO_ID}"

def query_hf_api(prompt):
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 450,
            "temperature": 0.8,
            "top_p": 0.92,
            "repetition_penalty": 1.2,
            "do_sample": True,
        }
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

prompt = "<|begin|>\nCategory: fantasy\nChild: Hana\n\nPage 1:"
result = query_hf_api(prompt)
print("API Response:")
print(result)

---

## Done!

Your custom story model is trained and ready. Next steps:

1. **Copy your HuggingFace repo ID** (e.g., `yourusername/kidlearn-story-model`)
2. **Add to your backend `.env`**:
   ```
   HF_MODEL_ID=yourusername/kidlearn-story-model
   HF_API_TOKEN=hf_your_token_here
   ```
3. **The backend will auto-detect** these env vars and use your custom model instead of Google AI

The model is only ~130MB and generates stories in seconds via the free HuggingFace Inference API.